# Exploratory Data Analysis (EDA)

## Credit Risk Probability Model for Alternative Data

### Task 2 — Exploratory Data Analysis

This notebook performs exploratory data analysis on the Xente transaction dataset to understand data quality, customer behavior, feature distributions, and relationships between variables before feature engineering and model development.

## 1. Importing Libraries

The following libraries are used for data manipulation, visualization, and exploratory analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")
%matplotlib inline

print("Libraries imported successfully!")

## 2. Loading the Dataset

The dataset is loaded from the raw data directory for initial exploration and quality assessment.

In [ ]:
# Import pandas
import pandas as pd

# Load dataset
df = pd.read_csv("../data/raw/data.csv")

# Display first 5 rows
df.head()

### Observation

The dataset contains transaction-level financial records with both numerical and categorical features. Initial inspection confirms the presence of customer identifiers, transaction values, product information, and transaction timestamps that may later be useful for feature engineering and behavioral analysis.

## 3. Dataset Overview

This section explores the overall structure of the dataset, including the number of rows, columns, feature names, and data types.

In [ ]:
# Dataset shape
print("Dataset Shape:", df.shape)

# Column names
print("\nColumns:")
print(df.columns)

# Dataset information
print("\nDataset Info:")
df.info()

### Observation

The dataset contains both numerical and categorical variables. Features such as Amount, Value, and CountryCode are numerical, while ProductCategory, ChannelId, and CurrencyCode are categorical. TransactionStartTime appears as an object datatype and may require datetime conversion during preprocessing.

## 4. Summary Statistics

Summary statistics provide an overview of the central tendency, spread, and distribution of numerical features in the dataset.

In [ ]:
# Summary statistics with transpose for readability
df.describe().T

### Observation 
High skewness/outliers in Amount & Value; zero variance in CountryCode; highly imbalanced target (0.2% fraud).


## 5. Distribution of Numerical Features

Visualizing numerical features helps identify the shape of the data distribution, skewness, concentration of values, and potential outliers. Financial transaction datasets often contain highly skewed distributions due to the presence of extreme transaction values.

In [ ]:
# Select numerical columns
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns

# Display numerical columns
print("Numerical Features:")
print(numerical_columns)

### Observation

The dataset contains several numerical variables related to transaction values, country information, pricing strategies, and fraud indicators. These features will be analyzed further to understand their distributions and variability.

In [ ]:
# Plot histograms for numerical features
df[numerical_columns].hist(figsize=(15,10), bins=30)

plt.suptitle("Distribution of Numerical Features", fontsize=16)
plt.tight_layout()
plt.show()

### Observation: Numerical Distributions

* **CountryCode & FraudResult:** Both appear as single, unvarying bars, reflecting a constant feature (256) and a heavily imbalanced target (0.2% fraud).
* **Amount & Value:** Heavily right-skewed and concentrated near zero, confirming the presence of extreme high-value outliers.
* **PricingStrategy:** Behaves as a discrete/categorical variable, dominated by strategies `2` and `4`.

### Distribution of Transaction Amount

In [ ]:
# Plot distribution of transaction amounts
plt.figure(figsize=(10,5))

sns.histplot(df["Amount"], kde=True)

plt.title("Distribution of Transaction Amount")
plt.xlabel("Transaction Amount")
plt.ylabel("Frequency")

plt.show()

### Observation: Distribution of Transaction Amount

* **Severe Right Skew:** The distribution is dominated by a sharp spike near zero (typical low-value transactions) with an extremely long tail extending up to $10,000,000$ (high-value outliers).
* **Negative Values:** A small, flat tail extends to $-1,000,000$, representing debits, chargebacks, or refunds.

### Distribution of Transaction Value

In [ ]:
# Plot distribution of transaction values
plt.figure(figsize=(10,5))

sns.histplot(df["Value"], kde=True)

plt.title("Distribution of Transaction Value")
plt.xlabel("Transaction Value")
plt.ylabel("Frequency")

plt.show()

### Skewness Analysis

In [ ]:
# Calculate skewness of numerical features
skewness = df[numerical_columns].skew().sort_values(ascending=False)

# Display skewness
skewness

### Observation: Skewness & Value Distributions

* **Extreme Positive Skewness:** Both `Value` (skewness ~51.29) and `Amount` (skewness ~51.10) exhibit extreme positive skewness. This mathematically confirms that their distributions are dominated by a massive volume of small transactions with a very long right tail of high-value outliers.
* **Target Imbalance:** `FraudResult` has a skewness of ~22.20, highlighting the severe class imbalance (0.2% fraud transactions) that will require class-balancing techniques during model training.
* **Constant Feature:** `CountryCode` has a skewness of 0.0, reflecting its completely uniform/constant value (256) which makes it irrelevant for predictive modeling.

## 6. Distribution of Categorical Features

Understanding the distribution of categorical features reveals structural patterns in customer behavior, payment channels, and service providers. We will analyze `ProductCategory`, `ChannelId`, `ProviderId`, and `PricingStrategy` using frequency tables and count plots.

In [ ]:
# Define categorical features to explore
categorical_cols = ['ProductCategory', 'ChannelId', 'ProviderId', 'PricingStrategy']

# Set up figure for countplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, ax=axes[i], order=order, palette='viridis')
    axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45 if col == 'ProductCategory' else 0)

plt.tight_layout()
plt.show()

# Print frequency tables for each feature
for col in categorical_cols:
    counts = df[col].value_counts()
    pcts = df[col].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts.round(2)})
    print(f"\nFrequency Table for {col}:")
    print(summary_df)

### Observation: Categorical Features

* **ProductCategory Concentration:** Over **94.5%** of transactions are either `financial_services` (47.47%) or `airtime` (47.07%). The remaining categories (utility bills, data bundles, etc.) represent tiny fragments of the dataset.
* **Dominant Channels:** Channels `ChannelId_3` (59.52%) and `ChannelId_2` (38.83%) account for **98.35%** of all transaction activity.
* **Provider Shares:** `ProviderId_4` (39.92%) and `ProviderId_6` (35.74%) process three-quarters of the platform's transactions.
* **PricingStrategy:** More than **83%** of transactions are billed under Pricing Strategy `2`, with Strategy `4` being the second most frequent.

## 7. Correlation Analysis

I examine the linear relationships between numerical variables to identify collinearity (redundancy) and gauge how features relate to the binary `FraudResult` target. Constant features (like `CountryCode`) are excluded.

In [ ]:
# Select numeric features with variance (excluding CountryCode)
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != 'CountryCode']
corr_matrix = df[numeric_cols].corr()

# Generate heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".4f", vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Observation: Correlation Analysis

* **High Multi-Collinearity:** There is a near-perfect correlation ($r \approx 0.9897$) between `Amount` and `Value`. Since `Value` is the absolute magnitude of `Amount`, they contain highly redundant information. For linear modeling (e.g., Logistic Regression), one of them should be excluded to avoid collinearity instability.
* **Strong Fraud Correlation:** transaction `Amount` ($r \approx 0.5574$) and `Value` ($r \approx 0.5667$) show moderately strong positive correlations with `FraudResult`. This suggests that fraudulent activity on this platform is heavily clustered in higher-value transactions.
* **PricingStrategy:** Has almost zero linear correlation with the other variables, representing a distinct charging model dimension.

## 8. Identifying Missing Values

Evaluating missing data is a critical quality step. Missing features require imputation strategy planning prior to feeding them into machine learning algorithms.

In [ ]:
# Calculate missing counts and percentages
missing_counts = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_pct.round(4)
})
missing_df

### Observation: Missing Values

* **Zero Missing Data:** The dataset has **0 missing values** across all columns. The data pipeline is highly reliable at logging details, meaning we do not need to implement any imputation strategy.

## 9. Outlier Detection

Financial values are highly prone to outliers. We use boxplots with log and symmetric log scales to inspect these outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Amount boxplot (uses symlog to display negative amounts/debits correctly)
sns.boxplot(data=df, y='Amount', ax=axes[0], color='skyblue')
axes[0].set_yscale('symlog')
axes[0].set_title('Boxplot of Transaction Amount (Symmetric Log Scale)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Amount (UGX)')

# Value boxplot
sns.boxplot(data=df, y='Value', ax=axes[1], color='salmon')
axes[1].set_yscale('log')
axes[1].set_title('Boxplot of Transaction Value (Log Scale)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Value (UGX)')

plt.tight_layout()
plt.show()

### Observation: Outlier Detection

* **Extreme Financial Spans:** The median transaction is small ($1,000$ UGX), but outliers span up to **$9,880,000$ UGX** on the positive side, and down to **$-1,000,000$ UGX** on the negative side.
* **Domain Implications:** In fintech/lending contexts, these high-value outliers are often legitimate large purchases or indicators of fraud. We should avoid arbitrary trimming/capping; instead, robust scaling (e.g., using `RobustScaler` or log transforms) is essential for linear models.

## 10. Key Insights

Based on the completed Exploratory Data Analysis, the following findings will shape our feature engineering and modeling decisions:

1. **Collinearity of Financial Features:** `Amount` and `Value` are almost perfectly correlated ($r \approx 0.9897$). To prevent model instability in linear classifiers, I should train on only one of them or engineer a sign feature (to capture negative vs. positive transactions) before discarding one.
2. **Zero-Variance Features:** `CountryCode` (constant at 256) and `CurrencyCode` (constant at 'UGX') have zero variance and must be excluded from feature variables.
3. **High Concentration:** The dataset is highly concentrated. Two product categories (`financial_services` and `airtime`) account for **94.5%** of all rows, and two channels represent **98.3%** of all transactions. Group-based aggregation features on these categories will likely yield high predictive value.
4. **Severe Class Imbalance:** Fraud cases account for only **0.2%** of the dataset. Traditional metrics like accuracy will be misleading; I must optimize for Precision-Recall AUC or F1-score and use rescaled class weights during training.
5. **High-Value Fraud Cluster:** There is a moderate-to-strong positive correlation ($r \approx 0.56$) between transaction size and `FraudResult`. Outlier detection and robust scaling will be crucial because fraud occurs predominantly in the extreme right tail of the distributions.